# Stage 8 — Postmortem & Final Report

Aggregates results from all stages and produces the final honest assessment.

Whether the strategy passed or failed, the discipline is the deliverable. If failed, an entry is added to `results/POSTMORTEM.md` documenting the failure mode for future research.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

RESULTS = Path('../results')
DATA_PROC = Path('../data/processed')

In [ ]:
chosen = json.loads((DATA_PROC / 'chosen_config.json').read_text())
holdout = json.loads((RESULTS / 'tables' / 'holdout_result.json').read_text())
multi = pd.read_csv(RESULTS / 'tables' / 'multi_coin_results.csv')
is_log = pd.read_csv(RESULTS / 'tables' / 'is_search_log.csv')

print(f'IS trials:              {len(is_log):,}')
print(f'Chosen config: ')
print(json.dumps(chosen['params'], indent=2))

In [ ]:
stages = pd.DataFrame([
    {'Stage': '0 Hypothesis',  'Status': 'LOCKED', 'Note': 'Pre-commitment signed'},
    {'Stage': '1 Data + Split','Status': 'CLEAN',  'Note': f'37,225 BTCUSDT 1h bars, 0 missing'},
    {'Stage': '2 IS Search',   'Status': 'DONE',   'Note': f'{len(is_log):,} trials evaluated'},
    {'Stage': '3 OOS Select',  'Status': 'DONE',   'Note': f'IS Sharpe {chosen["is_sharpe"]:.2f} → OOS {chosen["oos_sharpe"]:.2f} (retention {chosen["pf_retention"]:.0%})'},
    {'Stage': '4 Walk-forward','Status': 'see 05', 'Note': '6m rolling, ≥80% months PF>1 required'},
    {'Stage': '5 Holdout',     'Status': 'PASS' if holdout['gates_passed'] else 'FAIL', 'Note': f'PF {holdout["pf"]:.2f} / Sharpe {holdout["sharpe"]:.2f} / DD {holdout["max_dd"]:.1%} / DSR {holdout["dsr"]:.2f}'},
    {'Stage': '6 Multi-coin',  'Status': f'{int(multi["passes"].sum())}/{len(multi)}', 'Note': 'pass Stage 5 thresholds'},
])
stages

In [ ]:
stage5_pass = holdout['gates_passed']
stage6_pass = int(multi['passes'].sum()) >= 4
overall = stage5_pass and stage6_pass

print('=' * 60)
print('FINAL VERDICT')
print('=' * 60)
print(f'Stage 5 holdout:     {"PASS" if stage5_pass else "FAIL"}')
print(f'Stage 6 multi-coin:  {"PASS" if stage6_pass else "FAIL"}')
print(f'Overall:             {"STRATEGY VALIDATED" if overall else "STRATEGY RETIRED — see POSTMORTEM.md"}')
print()
if not overall:
    print('Per METHODOLOGY.md, the strategy will not advance to paper trading.')
    print('The negative result IS the deliverable — it demonstrates that the SOP')
    print('caught what would otherwise have been a losing live deployment.')